# Task Decomposition & Planning Architectures
This notebook demonstrates how to implement the State-of-the-Art **Plan-and-Execute** architecture using industry-standard SDKs: `pydantic` for schema enforcement, and `langgraph` for the state-machine orchestration.

**Dependencies required to run this code in production:** 
`pip install langgraph langchain langchain-openai pydantic`


## 1. Pydantic Models for Task Decomposition
We must force the "Planner" agent to output a strict array of tasks. We do this using Pydantic.


In [1]:
from pydantic import BaseModel, Field
# 1. Define the atomic unit of work
import sys, os; sys.path.insert(0, os.path.join(os.getcwd(), 'curriculum/intermediate/08-planning-task-decomposition')); from policy import SubTask, Plan
# Simulating the Planner LLM returning structured JSON:
llm_planner_output = {
    "subtasks": [
        {"task_id": 1, "description": "Query database for user 991's email.", "expected_tool": "database_query"},
        {"task_id": 2, "description": "Draft an apology email.", "expected_tool": "calculator"},
        {"task_id": 3, "description": "Send the drafted email.", "expected_tool": "web_search"}
    ]
}
parsed_plan = Plan(**llm_planner_output)
print("🧠 [Planner] Successfully decomposed goal into a strict Plan:")
for task in parsed_plan.subtasks:
    print(f"  Step {task.task_id}: {task.description} (Use: {task.expected_tool})")


🧠 [Planner] Successfully decomposed goal into a strict Plan:
  Step 1: Query database for user 991's email. (Use: database_query)
  Step 2: Draft an apology email. (Use: calculator)
  Step 3: Send the drafted email. (Use: web_search)


## 2. Implementing the State Graph (LangGraph)
We define a State object that holds the `plan` (the remaining tasks) and the `past_steps` (the scratchpad of results). 
Then, we wire up the Planner, Worker, and Re-Planner nodes.


In [2]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END

# 1. Define the Rolling State
class PlanExecuteState(TypedDict):
    input: str
    plan: list[SubTask]
    # operator.add ensures that when a node returns past_steps, they are APPENDED, not overwritten
    past_steps: Annotated[list[tuple], operator.add] 
    final_response: str

# 2. Define the Nodes
def planner_node(state: PlanExecuteState):
    print("\n[Node: Planner] Decomposing complex goal...")
    # Mock LLM generating the plan
    return {"plan": parsed_plan.subtasks}

def worker_node(state: PlanExecuteState):
    # Pop the first task
    current_task = state["plan"][0]
    print(f"\n👷 [Node: Worker] Assigned Task: {current_task.description}")
    
    # Run the bounded ReAct loop here (mocked for demo)
    print("   Worker is executing... ✅ Success.")
    result = f"Result of {current_task.task_id} completed successfully."
    
    # We remove the completed task from the plan, and append the result to the scratchpad
    return {
        "plan": state["plan"][1:],
        "past_steps": [(current_task.description, result)]
    }

def replanner_node(state: PlanExecuteState):
    print("🔄 [Node: Re-Planner] Evaluating worker's result against the remaining plan...")
    if len(state["plan"]) == 0:
        print("   ✅ Plan complete. Formulating final response to user.")
        return {"final_response": "All tasks completed successfully."}
    else:
        print(f"   ⏳ {len(state['plan'])} tasks remaining. Proceeding.")
        # In reality, this LLM call could rewrite the plan based on failure!
        return {"plan": state["plan"]}

# 3. Define the Router Edge
def route_step(state: PlanExecuteState):
    if "final_response" in state:
        return END
    else:
        return "worker"

# 4. Compile the Graph
workflow = StateGraph(PlanExecuteState)

workflow.add_node("planner", planner_node)
workflow.add_node("worker", worker_node)
workflow.add_node("replanner", replanner_node)

workflow.set_entry_point("planner")
workflow.add_edge("planner", "worker")
workflow.add_edge("worker", "replanner")
workflow.add_conditional_edges("replanner", route_step, {"worker": "worker", END: END})

app = workflow.compile()
print("✅ LangGraph Plan-and-Execute StateMachine compiled successfully.\n")


✅ LangGraph Plan-and-Execute StateMachine compiled successfully.



## 3. Execution Trace
Watch the graph loop dynamically. The Worker executes, the Re-Planner evaluates, and it loops back to the Worker until the plan is empty.


In [3]:
initial_state = {
    "input": "Find user 991 and email them an apology.",
    "plan": [],
    "past_steps": []
}

print("--- EXECUTING COMPLEX GOAL ---")
final_state = app.invoke(initial_state)

print("\n--- EXECUTION SUMMARY ---")
print("Final Response:", final_state.get("final_response"))
print("\nScratchpad (Past Steps):")
for step, result in final_state["past_steps"]:
    print(f" - {step}  =>  {result}")


--- EXECUTING COMPLEX GOAL ---

[Node: Planner] Decomposing complex goal...

👷 [Node: Worker] Assigned Task: Query database for user 991's email.
   Worker is executing... ✅ Success.
🔄 [Node: Re-Planner] Evaluating worker's result against the remaining plan...
   ⏳ 2 tasks remaining. Proceeding.

👷 [Node: Worker] Assigned Task: Draft an apology email.
   Worker is executing... ✅ Success.
🔄 [Node: Re-Planner] Evaluating worker's result against the remaining plan...
   ⏳ 1 tasks remaining. Proceeding.

👷 [Node: Worker] Assigned Task: Send the drafted email.
   Worker is executing... ✅ Success.
🔄 [Node: Re-Planner] Evaluating worker's result against the remaining plan...
   ✅ Plan complete. Formulating final response to user.

--- EXECUTION SUMMARY ---
Final Response: All tasks completed successfully.

Scratchpad (Past Steps):
 - Query database for user 991's email.  =>  Result of 1 completed successfully.
 - Draft an apology email.  =>  Result of 2 completed successfully.
 - Send the dra